# Hướng Dẫn Giải Thích Chi Tiết: `src/features.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của lớp biến đổi dữ liệu `DataTransformer` trong `src/features.py`.

---

## 🔍 1. Tại Sao Cần Biến Đổi Dữ Liệu Thời Gian (Time-Series)?

Mô hình học máy truyền thống như hồi quy tuyến tính nhận đầu vào dạng bảng 2D `[số mẫu, số đặc trưng]`. Tuy nhiên, để các mô hình mạng Deep Learning (như Transformer) học được mối quan hệ tuần tự của giá cổ phiếu qua nhiều ngày liên tiếp, chúng ta cần biến đổi dữ liệu thành cấu trúc **3D**:
$$\text{Shape: } [\text{Số mẫu}, \text{Số bước thời gian (Lookback Window)}, \text{Số đặc trưng}]$$

Đồng thời, giá trị của các chỉ báo kỹ thuật có khoảng biến thiên rất khác nhau (ví dụ: RSI từ 0 đến 100, Khối lượng giao dịch từ hàng chục nghìn đến hàng triệu). Do đó, việc co giãn dữ liệu về khoảng `[0, 1]` là bắt buộc để mô hình tối ưu hội tụ nhanh.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))

from src.data_loader import fetch_and_prepare_data
from src.features import DataTransformer
print("Import lớp DataTransformer thành công!")

## ⚙️ 2. Các Bước Xử Lý Của Lớp `DataTransformer`

### Khởi tạo đối tượng
- `time_steps`: Kích thước cửa sổ trượt (lookback window), hiện tại được nâng cấp lên **45 ngày**.
- `feature_scaler`: Đối tượng `MinMaxScaler(feature_range=(0, 1))` chuẩn hóa đầu vào.
- `target_scaler`: Đối tượng `MinMaxScaler(feature_range=(0, 1))` chuẩn hóa đầu ra.

In [ ]:
# Khởi tạo transformer
transformer = DataTransformer(time_steps=45)
print(f"Lookback window (time_steps): {transformer.time_steps}")
print(f"Danh sách các cột đặc trưng đầu vào ({len(transformer.feature_cols)} cột):")
print(transformer.feature_cols)

### Hàm `fit_transform_data(df)`
Hàm này tính toán:
1. **Biến mục tiêu (Target):** Lợi suất mở cửa kế tiếp so với giá đóng cửa hôm nay:
   $$\text{Target}_t = \frac{\text{Open}_{t} - \text{Close}_{t-1}}{\text{Close}_{t-1}}$$
2. **Đặc trưng (Features):** Trích xuất 15 đặc trưng bao gồm các chỉ báo kỹ thuật và tỷ suất sinh lời thị trường vĩ mô (`market_return`), co giãn tất cả về khoảng `[0, 1]`.

In [ ]:
# Tải dữ liệu thật
df_real = fetch_and_prepare_data("VNM.VN", start_date="2024-01-01", end_date="2024-12-31")

# Áp dụng fit_transform_data
X_scaled, y_scaled = transformer.fit_transform_data(df_real)
print(f"Đặc trưng sau chuẩn hóa (shape): {X_scaled.shape}")
print(f"Giá trị đặc trưng lớn nhất: {X_scaled.max()}; Nhỏ nhất: {X_scaled.min()}")
print(f"Biến mục tiêu sau chuẩn hóa (shape): {y_scaled.shape}")

### Hàm `create_sliding_windows(X_data, y_data)`
Hàm này quét qua dữ liệu dạng bảng 2D và gom các nhóm gồm 45 phiên liên tiếp tạo thành cấu trúc 3D.
Ví dụ, nếu chúng ta muốn dự báo ngày thứ 46, đầu vào sẽ là thông tin từ ngày 1 đến ngày 45.

In [ ]:
# Tạo sliding window dạng 3D
X_3D, y_3D = transformer.create_sliding_windows(X_scaled, y_scaled)
print(f"Hình dạng dữ liệu 3D (X_3D): {X_3D.shape}")
print(f"Hình dạng mục tiêu tương ứng (y_3D): {y_3D.shape}")
print(f"Giải thích: Có {X_3D.shape[0]} mẫu dự báo, mỗi mẫu là một chuỗi gồm {X_3D.shape[1]} ngày liên tiếp, mỗi ngày chứa {X_3D.shape[2]} đặc trưng kỹ thuật và vĩ mô.")

### Hàm `split_train_test_chronological(df, X_3D, y_3D, train_ratio=0.8)`

**QUAN TRỌNG:** Trong dữ liệu chuỗi thời gian, ta **không bao giờ** được phân chia tập huấn luyện (train) và kiểm thử (test) ngẫu nhiên (như dùng `train_test_split` của Sklearn). Làm như vậy sẽ gây ra hiện tượng **rò rỉ dữ liệu tương lai** (Data Leakage) vào quá khứ.

Hàm này thực hiện chia cắt tuần tự theo thời gian thực tế: lấy 80% thời gian đầu để huấn luyện và 20% thời gian sau để kiểm thử.

In [ ]:
X_train, y_train, X_test, y_test, y_test_raw = transformer.split_train_test_chronological(
    df_real, X_3D, y_3D, train_ratio=0.8
)

print(f"Số lượng mẫu tập Train: {X_train.shape[0]}")
print(f"Số lượng mẫu tập Test : {X_test.shape[0]}")